In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack, csr_matrix
from sklearn.preprocessing import StandardScaler
import joblib

In [2]:
df = pd.read_csv('data/AI_Resume_Screening.csv')
df = pd.DataFrame(df)


# Drop unnecessary columns
cols_to_drop = [
    "Resume_ID",
    "Name",
    "Score (0-100)",
    "Salary Expectation ($)"
]

df = df.drop(columns=cols_to_drop)

# Fill missing text
text_cols = ['Skills', 'Education', 'Certifications', 'Job Role']
for col in text_cols:
    df[col] = df[col].fillna('')

# Encode target
df['Recruiter Decision'] = df['Recruiter Decision'].map({
    'Reject': 0,
    'Hire': 1
})

In [3]:
# TF-IDF
tfidf_skills = TfidfVectorizer(max_features=300, ngram_range=(1,2))
X_skills = tfidf_skills.fit_transform(df['Skills'])

tfidf_edu = TfidfVectorizer(max_features=20)
X_edu = tfidf_edu.fit_transform(df['Education'])

tfidf_cert = TfidfVectorizer(max_features=50)
X_cert = tfidf_cert.fit_transform(df['Certifications'])

tfidf_job = TfidfVectorizer(max_features=30)
X_job = tfidf_job.fit_transform(df['Job Role'])

# Numeric features → sparse
X_num = df[['Experience (Years)',
            'Projects Count']].values

X_num_sparse = csr_matrix(X_num)
scaler = StandardScaler()
X_num_scaled = scaler.fit_transform(X_num)
X_num_sparse = csr_matrix(X_num_scaled)


# Combine tất cả (giữ sparse)
X_all = hstack([
    X_skills,
    X_edu,
    X_cert,
    X_job,
    X_num_sparse
])

y = df['Recruiter Decision']

print("Shape X:", X_all.shape)
print("Ready for training (sparse format)")

Shape X: (1000, 79)
Ready for training (sparse format)


In [4]:
df.to_csv(
    "cleaned_data.csv",
    index=False,
    encoding="utf-8"
)
print(" Đã tạo cleaned_data.csv")

 Đã tạo cleaned_data.csv


In [5]:
df

,Skills,Experience (Years),Education,Certifications,Job Role,Recruiter Decision,Projects Count
0,"TensorFlow, NLP, Pytorch",10,B.Sc,,AI Researcher,1,8
1,"Deep Learning, Machine Learning, Python, SQL",10,MBA,Google ML,Data Scientist,1,1
2,"Ethical Hacking, Cybersecurity, Linux",1,MBA,Deep Learning Specialization,Cybersecurity Analyst,1,7
3,"Python, Pytorch, TensorFlow",7,B.Tech,AWS Certified,AI Researcher,1,0
4,"SQL, React, Java",4,PhD,,Software Engineer,1,9
...,...,...,...,...,...,...,...
995,"Cybersecurity, Linux, Ethical Hacking",0,B.Sc,,Cybersecurity Analyst,0,9
996,"Deep Learning, Machine Learning",0,MBA,Deep Learning Specialization,Data Scientist,0,5
997,"TensorFlow, NLP",0,B.Tech,Google ML,AI Researcher,1,9
998,"Linux, Networking, Cybersecurity, Ethical Hacking",8,PhD,AWS Certified,Cybersecurity Analyst,1,10


In [6]:
joblib.dump(X_all, "pkl/X_sparse.pkl")
joblib.dump(y, "pkl/y.pkl")

joblib.dump(tfidf_skills, "pkl/tfidf_skills.pkl")
joblib.dump(tfidf_edu, "pkl/tfidf_edu.pkl")
joblib.dump(tfidf_cert, "pkl/tfidf_cert.pkl")
joblib.dump(tfidf_job, "pkl/tfidf_job.pkl")

joblib.dump(scaler, "pkl/scaler.pkl")

print("Saved full vectorization pipeline")

Saved full vectorization pipeline
